# 🚀 Marvedge — Task-00077: Ultra-Fast Simultaneous GPU Benchmark

Runs the **V4 optimised pipeline** (I-frame-only keyframe detection + YuNet ONNX + track-merge + single-FFmpeg-cmd crop) **simultaneously** on 3 videos of different lengths.

This is the same pipeline that processed an **11.36 GB / 22-min video in 4.5 min** (8x real-time).

**Before running:** `Runtime → Change runtime type → GPU (T4 or L4)`

In [ ]:
# Cell 1 — verify GPU
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# Cell 2 — install deps (YuNet ONNX, no TalkNet-ASD needed!)
!pip install -q opencv-python-headless scipy tqdm
# Download YuNet ONNX model (tiny 1 MB CPU detector — the V4 magic)
import urllib.request, os
YUNET_PATH = '/content/face_detection_yunet_2023mar.onnx'
if not os.path.exists(YUNET_PATH):
    urllib.request.urlretrieve(
        'https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx',
        YUNET_PATH
    )
    print('YuNet model downloaded.')
print('Ready.')

In [ ]:
# Cell 3 — Mount Google Drive and select videos
import os, shutil
from google.colab import drive

# 1. Mount Google Drive (this will ask for permission)
drive.mount('/content/drive')

os.makedirs('/content/clips', exist_ok=True)

# 2. Define the paths to your 3 videos in Google Drive here:
#    (Change these paths to where your actual videos are stored!)
DRIVE_VIDEOS = [
    '/content/drive/MyDrive/your_40s_video.mp4',
    '/content/drive/MyDrive/your_5min_video.mp4',
    '/content/drive/MyDrive/your_20min_video.mp4'
]

VIDEO_PATHS = []
for drive_path in DRIVE_VIDEOS:
    if os.path.exists(drive_path):
        name = os.path.basename(drive_path)
        dest = f'/content/clips/{name}'
        print(f'Copying {name} from Drive...')
        shutil.copy2(drive_path, dest)
        VIDEO_PATHS.append(dest)
        print(f'  ✅ {name} ready at {dest}')
    else:
        print(f'  ⚠️ Skipping (File not found): {drive_path}')

print(f'
{len(VIDEO_PATHS)} video(s) ready for benchmarking.')


In [ ]:
# Cell 4 — the V4 ultra-fast pipeline engine
# Identical architecture to the one that ran 11.36 GB in 4.5 min:
#   S1: ffmpeg audio extract
#   S2+S3: -skip_frame nokey I-frames only → YuNet ONNX detect
#   S4: scipy interp1d interpolation
#   S5: IoU tracker + spatial-merge (1527 tracks → ~10-30)
#   S6: single ffmpeg crop command per track

import sys, os, time, json, subprocess, threading
import numpy as np
import cv2
import tqdm
from scipy.interpolate import interp1d
from shutil import rmtree

TARGET_FPS = 25
T4_COST_PER_SEC = 0.35 / 3600
L4_COST_PER_SEC = 0.80 / 3600

# ── GPU poller ───────────────────────────────────────────────────────────────
def poll_gpu(stop_evt, samples):
    cmd = ['nvidia-smi',
           '--query-gpu=memory.used,memory.total,utilization.gpu,temperature.gpu',
           '--format=csv,noheader,nounits']
    while not stop_evt.is_set():
        try:
            raw = subprocess.check_output(cmd, stderr=subprocess.DEVNULL, timeout=2)
            p = raw.decode().strip().split(',')
            if len(p) >= 4:
                samples.append({'mem': int(p[0]), 'tot': int(p[1]),
                                 'util': int(p[2]), 'temp': int(p[3])})
        except: pass
        stop_evt.wait(0.1)

# ── Video info ───────────────────────────────────────────────────────────────
def get_video_info(path):
    try:
        raw = subprocess.check_output(
            ['ffprobe','-v','error','-select_streams','v:0',
             '-show_entries','stream=width,height,r_frame_rate,nb_frames',
             '-show_entries','format=duration,size','-of','json', path],
            stderr=subprocess.DEVNULL).decode()
        d = json.loads(raw)
        st = d.get('streams',[{}])[0]
        fmt = d.get('format',{})
        w, h = int(st.get('width',1920)), int(st.get('height',1080))
        rfr = st.get('r_frame_rate','25/1')
        n, dn = rfr.split('/') if '/' in rfr else (rfr,'1')
        fps = float(n)/max(float(dn),1e-6)
        dur = float(st.get('duration') or fmt.get('duration') or 0)
        nb = st.get('nb_frames')
        total = int(nb) if nb and nb != 'N/A' else int(dur*fps)
        size_bytes = int(fmt.get('size', 0))
        return dict(width=w,height=h,fps=fps,duration=dur,total_frames=total,size_bytes=size_bytes)
    except Exception as e:
        return dict(width=1920,height=1080,fps=25,duration=0,total_frames=0,size_bytes=0)

# ── S1: Audio extract ────────────────────────────────────────────────────────
def stage_audio(video_path, audio_path, threads=4):
    subprocess.run(['ffmpeg','-y','-i',video_path,'-vn','-ac','1','-ar','16000',
                    '-acodec','pcm_s16le','-threads',str(threads),audio_path,
                    '-loglevel','panic'], check=True)

# ── S2+S3: Keyframe detection ─────────────────────────────────────────────────
def _stderr_reader(pipe, timestamps):
    for line in pipe:
        line = line.decode('utf-8', errors='replace')
        if 'pts_time:' in line:
            for part in line.split():
                if part.startswith('pts_time:'):
                    try: timestamps.append(float(part.split(':')[1]))
                    except: pass
                    break
    pipe.close()

def stage_detect(video_path, yunet_path, det_w, det_h, native_w, native_h, duration, threads=4):
    frame_size = det_w * det_h * 3
    scale_x, scale_y = native_w/det_w, native_h/det_h
    vf = f'scale={det_w}:{det_h}:flags=bilinear,showinfo'
    cmd = ['ffmpeg','-skip_frame','nokey','-flags2','fast','-threads','0',
           '-i',video_path,'-vf',vf,'-f','rawvideo','-pix_fmt','rgb24','-vsync','vfr','-']
    detector = cv2.FaceDetectorYN.create(yunet_path,'',(det_w,det_h),
                                          score_threshold=0.5,nms_threshold=0.3,top_k=5000)
    est_kf = max(10, int(duration/3))
    proc = subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.PIPE,bufsize=frame_size*4)
    timestamps = []
    t = threading.Thread(target=_stderr_reader,args=(proc.stderr,timestamps),daemon=True)
    t.start()
    sparse_dets = {}
    fidx = 0
    try:
        with tqdm.tqdm(total=est_kf,desc='  S2+S3 detect',unit='kf') as pbar:
            while True:
                raw = proc.stdout.read(frame_size)
                if len(raw) < frame_size: break
                frame = np.frombuffer(raw,dtype=np.uint8).reshape(det_h,det_w,3)
                frame_bgr = cv2.cvtColor(frame,cv2.COLOR_RGB2BGR)
                _, faces = detector.detect(frame_bgr)
                ts = None
                for _ in range(50):
                    if fidx < len(timestamps):
                        ts = timestamps[fidx]; break
                    time.sleep(0.01)
                if ts is None: ts = fidx*(duration/max(est_kf,1))
                src_frame = int(ts*TARGET_FPS)
                fd = []
                if faces is not None:
                    for face in faces:
                        x,y,fw,fh = float(face[0]),float(face[1]),float(face[2]),float(face[3])
                        x1=max(0,min(int(x*scale_x),native_w))
                        y1=max(0,min(int(y*scale_y),native_h))
                        x2=max(0,min(int((x+fw)*scale_x),native_w))
                        y2=max(0,min(int((y+fh)*scale_y),native_h))
                        fd.append({'frame':src_frame,'timestamp':round(ts,4),'bbox':[x1,y1,x2,y2],'conf':round(float(face[-1]),3)})
                sparse_dets[fidx] = fd
                fidx += 1; pbar.update(1); pbar.total = max(pbar.total,fidx+1)
    finally:
        proc.stdout.close(); proc.wait(); t.join(timeout=5)
    return sparse_dets, fidx

# ── S4: Interpolation ─────────────────────────────────────────────────────────
def stage_interpolate(sparse_dets, duration, native_w, native_h):
    total = int(duration*TARGET_FPS)+TARGET_FPS
    dets = [[] for _ in range(total)]
    pi = sorted(sparse_dets.keys())
    for p in pi:
        for det in sparse_dets[p]:
            tf = det['frame']
            if 0 <= tf < total: dets[tf].append(det)
    for i in range(len(pi)-1):
        ds,de = sparse_dets[pi[i]],sparse_dets[pi[i+1]]
        if not ds or not de: continue
        tf_s,tf_e = ds[0]['frame'],de[0]['frame']
        gap = tf_e-tf_s
        if gap <= 1: continue
        matches=[]
        for ai,a in enumerate(ds):
            for bi,b in enumerate(de):
                xA=max(a['bbox'][0],b['bbox'][0]);yA=max(a['bbox'][1],b['bbox'][1])
                xB=min(a['bbox'][2],b['bbox'][2]);yB=min(a['bbox'][3],b['bbox'][3])
                inter=max(0,xB-xA)*max(0,yB-yA)
                aA=max(0,a['bbox'][2]-a['bbox'][0])*max(0,a['bbox'][3]-a['bbox'][1])
                aB=max(0,b['bbox'][2]-b['bbox'][0])*max(0,b['bbox'][3]-b['bbox'][1])
                denom=float(aA+aB-inter)
                iou = inter/denom if denom>0 else 0.0
                if iou >= 0.3: matches.append((iou,ai,bi))
        matches.sort(reverse=True)
        ua,ub=set(),set()
        for _,ai,bi in matches:
            if ai in ua or bi in ub: continue
            ua.add(ai);ub.add(bi)
            bb_s=np.array(ds[ai]['bbox'],dtype=float)
            bb_e=np.array(de[bi]['bbox'],dtype=float)
            for g in range(1,gap):
                alpha=g/gap;tf=tf_s+g
                if tf>=total: continue
                bb=(bb_s*(1-alpha)+bb_e*alpha).tolist()
                dets[tf].append({'frame':tf,'timestamp':round(tf/TARGET_FPS,4),'bbox':bb,'conf':0.9})
    return dets

# ── S5: Track + Merge ─────────────────────────────────────────────────────────
def stage_track(dets, min_track=10, num_failed=200):
    active,completed=[],[]
    for fidx,faces in enumerate(dets):
        still=[]
        for t in active:
            if fidx-t[-1]['frame'] > num_failed: completed.append(t)
            else: still.append(t)
        active=still
        if not faces: continue
        matches=[]
        for ti,t in enumerate(active):
            for di,d in enumerate(faces):
                a,b=d['bbox'],t[-1]['bbox']
                xA=max(a[0],b[0]);yA=max(a[1],b[1]);xB=min(a[2],b[2]);yB=min(a[3],b[3])
                inter=max(0,xB-xA)*max(0,yB-yA)
                aA=max(0,a[2]-a[0])*max(0,a[3]-a[1]);aB=max(0,b[2]-b[0])*max(0,b[3]-b[1])
                dn=float(aA+aB-inter)
                iou=inter/dn if dn>0 else 0.0
                if iou>=0.3: matches.append((iou,ti,di))
        matches.sort(reverse=True)
        ut,ud=set(),set()
        for _,ti,di in matches:
            if ti not in ut and di not in ud:
                active[ti].append(faces[di]);ut.add(ti);ud.add(di)
        for di,d in enumerate(faces):
            if di not in ud: active.append([d])
    completed.extend(active)
    raw_tracks=[]
    for raw in completed:
        if len(raw)<min_track: continue
        fn=np.array([f['frame'] for f in raw],dtype=float)
        bb=np.array([f['bbox'] for f in raw],dtype=float)
        uf,ui=np.unique(fn,return_index=True)
        if len(uf)<min_track: continue
        fn=uf;bb=bb[ui]
        fi=np.arange(int(fn[0]),int(fn[-1])+1)
        bi=np.stack([interp1d(fn,bb[:,j],bounds_error=False,fill_value=(bb[0,j],bb[-1,j]))(fi) for j in range(4)],axis=1)
        raw_tracks.append({'frame':fi,'bbox':bi,'is_fallback':False})
    # spatial merge
    if raw_tracks:
        centers=[(float(np.median((t['bbox'][:,0]+t['bbox'][:,2])/2)),float(np.median((t['bbox'][:,1]+t['bbox'][:,3])/2))) for t in raw_tracks]
        parent=list(range(len(raw_tracks)))
        def find(x):
            while parent[x]!=x: parent[x]=parent[parent[x]];x=parent[x]
            return x
        for i in range(len(raw_tracks)):
            for j in range(i+1,len(raw_tracks)):
                ci,cj=centers[i],centers[j]
                dist=((ci[0]-cj[0])**2+(ci[1]-cj[1])**2)**0.5
                avg_w=float(np.mean(raw_tracks[i]['bbox'][:,2]-raw_tracks[i]['bbox'][:,0]))
                if avg_w>0 and dist<avg_w*1.5:
                    ri,rj=find(i),find(j)
                    if ri!=rj:
                        li,lj=len(raw_tracks[ri]['frame']),len(raw_tracks[rj]['frame'])
                        parent[rj]=ri if li>=lj else ri
        groups={}
        for i in range(len(raw_tracks)): groups.setdefault(find(i),[]).append(i)
        raw_tracks=[raw_tracks[max(m,key=lambda i:len(raw_tracks[i]['frame']))] for m in groups.values()]
    raw_tracks.sort(key=lambda x:x['frame'][0])
    return raw_tracks

# ── S6: FFmpeg crop ───────────────────────────────────────────────────────────
def stage_crop(video_path, audio_path, tracks, pycrop, threads=4, native_w=1920, native_h=1080):
    results=[]
    for ii,track in enumerate(tracks):
        frames=track['frame'];bboxes=track['bbox']
        cx=float(np.median((bboxes[:,0]+bboxes[:,2])/2))
        cy=float(np.median((bboxes[:,1]+bboxes[:,3])/2))
        cs=float(np.median(np.maximum(bboxes[:,2]-bboxes[:,0],bboxes[:,3]-bboxes[:,1])/2))
        cw=int(cs*2*1.4);ch=int(cs*2*1.8)
        cx2=max(0,min(int(cx-cw/2),native_w-cw));cy2=max(0,min(int(cy-ch/2),native_h-ch))
        cw=min(cw,native_w-cx2);ch=min(ch,native_h-cy2)
        if cw<32 or ch<32: continue
        t0=float(frames[0])/TARGET_FPS;t1=float(frames[-1]+1)/TARGET_FPS
        out=f'{pycrop}/{ii:05d}.avi'
        cmd=['ffmpeg','-y','-ss',f'{t0:.4f}','-i',video_path,
             '-ss',f'{t0:.4f}','-i',audio_path,'-t',f'{t1-t0:.4f}',
             '-filter:v',f'crop={cw}:{ch}:{cx2}:{cy2},scale=224:224',
             '-r',str(TARGET_FPS),'-map','0:v:0','-map','1:a:0',
             '-c:v','libx264','-preset','ultrafast','-crf','23',
             '-c:a','copy','-threads',str(threads),out,'-loglevel','panic']
        subprocess.run(cmd,check=True)
        results.append(out)
    return results

print('V4 pipeline engine loaded ✅')
print('Key optimisations active:')
print('  ✓ -skip_frame nokey  → only decode I-frames (~300 instead of 64500)')
print('  ✓ YuNet ONNX         → fast CPU face detector, no GPU memory needed')
print('  ✓ Spatial track merge → 1527 tracks → ~10-30 clean tracks')
print('  ✓ Single FFmpeg crop  → 1 cmd per track, not 3')

In [ ]:
# Cell 5 — run all 3 videos SIMULTANEOUSLY using ThreadPoolExecutor
from concurrent.futures import ThreadPoolExecutor, as_completed

YUNET_PATH = '/content/face_detection_yunet_2023mar.onnx'
DET_SCALE  = 0.25
MIN_TRACK  = 10
NUM_FAILED = 200
THREADS    = 4

gpu_samples = []
gpu_stop = threading.Event()
gpu_thread = threading.Thread(target=poll_gpu, args=(gpu_stop, gpu_samples), daemon=True)
gpu_thread.start()

def run_v4_pipeline(video_path):
    basename = os.path.splitext(os.path.basename(video_path))[0]
    save_dir = f'/tmp/bench_{basename}'
    if os.path.exists(save_dir): rmtree(save_dir)
    pywork = f'{save_dir}/pywork'; pycrop = f'{save_dir}/pycrop'
    os.makedirs(pywork, exist_ok=True); os.makedirs(pycrop, exist_ok=True)
    audio_path = f'{save_dir}/audio.wav'

    vinfo = get_video_info(video_path)
    native_w,native_h = vinfo['width'],vinfo['height']
    duration = vinfo['duration']
    det_w = max(64, int(native_w*DET_SCALE))
    det_h = max(64, int(native_h*DET_SCALE))

    t_total = time.perf_counter()
    timings = {}

    t0=time.perf_counter(); stage_audio(video_path,audio_path,THREADS); timings['S1_audio']=round(time.perf_counter()-t0,2)
    t0=time.perf_counter()
    sparse_dets,n_kf = stage_detect(video_path,YUNET_PATH,det_w,det_h,native_w,native_h,duration,THREADS)
    timings['S2_S3_detect']=round(time.perf_counter()-t0,2)
    t0=time.perf_counter(); dets=stage_interpolate(sparse_dets,duration,native_w,native_h); timings['S4_interp']=round(time.perf_counter()-t0,2)
    t0=time.perf_counter(); tracks=stage_track(dets,MIN_TRACK,NUM_FAILED); timings['S5_track']=round(time.perf_counter()-t0,2)
    if not tracks:
        total_f=len(dets);x1=int(native_w*0.17);y1=int(native_h*0.17);x2=int(native_w*0.83);y2=int(native_h*0.83)
        frames=np.arange(total_f);bboxes=np.tile(np.array([x1,y1,x2,y2],dtype=float),(total_f,1))
        tracks=[{'frame':frames,'bbox':bboxes,'is_fallback':True}]
    t0=time.perf_counter(); crops=stage_crop(video_path,audio_path,tracks,pycrop,THREADS,native_w,native_h); timings['S6_crop']=round(time.perf_counter()-t0,2)

    total_elapsed = round(time.perf_counter()-t_total, 2)

    return {
        'video': os.path.basename(video_path),
        'duration_s': round(duration,1),
        'file_size_mb': round(vinfo['size_bytes']/1e6,1),
        'resolution': f'{native_w}x{native_h}',
        'total_frames': vinfo['total_frames'],
        'keyframes_decoded': n_kf,
        'tracks_found': len(tracks),
        'crops_written': len(crops),
        'wall_time_s': total_elapsed,
        'realtime_factor': round(duration/max(total_elapsed,0.001),2),
        'fps_throughput': round(vinfo['total_frames']/max(total_elapsed,0.001),1),
        'timings': timings,
    }

# ── Run all videos simultaneously ────────────────────────────────────────────
all_results = {}
suite_t0 = time.perf_counter()

print(f'Starting simultaneous benchmark on {len(VIDEO_PATHS)} video(s)...')
print('═'*65)

with ThreadPoolExecutor(max_workers=len(VIDEO_PATHS)) as executor:
    futures = {executor.submit(run_v4_pipeline, p): p for p in VIDEO_PATHS}
    for future in as_completed(futures):
        path = futures[future]
        try:
            result = future.result()
            all_results[path] = result
            print(f'  ✅ {result["video"]} done in {result["wall_time_s"]}s ({result["realtime_factor"]}x real-time)')
        except Exception as exc:
            print(f'  ❌ {path} failed: {exc}')
            all_results[path] = {'video': os.path.basename(path), 'error': str(exc)}

suite_elapsed = round(time.perf_counter()-suite_t0, 2)

gpu_stop.set(); gpu_thread.join()

max_vram = max((s['mem'] for s in gpu_samples), default=0)
vram_tot = gpu_samples[0]['tot'] if gpu_samples else 0
avg_util = round(sum(s['util'] for s in gpu_samples)/max(len(gpu_samples),1), 1)
max_temp = max((s['temp'] for s in gpu_samples), default=0)

print('═'*65)
print(f'Total suite time (parallel): {suite_elapsed}s')
print(f'GPU: peak VRAM={max_vram}/{vram_tot} MiB  avg_util={avg_util}%  max_temp={max_temp}°C')

In [ ]:
# Cell 6 — comparison table + cost projection
import pandas as pd

rows = []
for path, r in sorted(all_results.items(), key=lambda x: x[1].get('duration_s',0)):
    if 'error' in r:
        rows.append({'Video': r['video'], 'Length': 'ERROR', 'File MB': '-',
                     'Resolution': '-', 'Keyframes': '-', 'Tracks': '-',
                     'Wall-clock(s)': '-', 'x-Realtime': '-',
                     'FPS throughput': '-',
                     'T4 cost($)': '-', 'L4 cost($)': '-'})
        continue
    wt = r['wall_time_s']
    rows.append({
        'Video': r['video'],
        'Length': f"{r['duration_s']}s",
        'File MB': r['file_size_mb'],
        'Resolution': r['resolution'],
        'Keyframes decoded': r['keyframes_decoded'],
        'Tracks found': r['tracks_found'],
        'Wall-clock(s)': wt,
        'x-Realtime': r['realtime_factor'],
        'FPS throughput': r['fps_throughput'],
        'T4 cost($)': round(wt * T4_COST_PER_SEC, 5),
        'L4 cost($)': round(wt * L4_COST_PER_SEC, 5),
    })

df = pd.DataFrame(rows)
print('\n=== MARVEDGE V4 PIPELINE — SIMULTANEOUS BENCHMARK RESULTS ===')
print(df.to_string(index=False))

# Stage timing breakdown
print('\n=== STAGE TIMING BREAKDOWN ===')
for path, r in sorted(all_results.items(), key=lambda x: x[1].get('duration_s',0)):
    if 'error' in r: continue
    print(f'\n  {r["video"]} ({r["duration_s"]}s):')
    for stage, t in r['timings'].items():
        print(f'    {stage:<20}: {t}s')

# Save results
os.makedirs('/content/benchmark_results', exist_ok=True)
with open('/content/benchmark_results/simultaneous_results.json','w') as f:
    json.dump({'suite_elapsed_s': suite_elapsed,
               'gpu': {'max_vram_mib': max_vram, 'vram_total_mib': vram_tot,
                       'avg_util_pct': avg_util, 'max_temp_c': max_temp},
               'results': list(all_results.values())}, f, indent=2, default=str)
print('\n✅ Results saved to /content/benchmark_results/simultaneous_results.json')